# 05 - בידוד של תחנות קריטיות

מדד betweenness centrality גבוה מלמד אותנו שתחנה נושאת עליה מסלולים קצרים רבים ברשת -
אך הוא **אינו** מלמד שאובדן התחנה אכן היה מותיר מישהו מנותק. אם תחנה אחרת ממוקמת
80 מטר במורד הרחוב, הנוסע פשוט הולך אליה ברגל והשכונה נותרת מחוברת. הנקודות השבירות
באמת הן התחנות שהן גם קריטיות *מבחינה מבנית* **וגם** מבודדות *מבחינה מרחבית*: ללא
חלופה במרחק הליכה.

מחברת זו היא "מבחן המציאות" של תוצאות הצנטרליות. היא נוטלת את המדדים לכל תחנה שהופקו
בשלב הצנטרליות, מגדירה **קריטית = העשירון העליון (top 10%) של betweenness**, ולאחר מכן
משתמשת ב-`BallTree` עם מטריקת haversine כדי למדוד, עבור כל תחנה, את מרחק המעגל הגדול
אל התחנה השכנה הקרובה ביותר. תחנות קריטיות שהחלופה הקרובה ביותר אליהן רחוקה מ-**300 מ'**
מסומנות כנקודות כשל בודדות ומבודדות.

**שאלת המחקר:** *מבין התחנות שהגרף מגדיר כקריטיות, כמה מהן קריטיות בפועל - כלומר,
נטולות תחליף בר-הליכה?*

### קלט
- `outputs/nb/04_centrality_analysis/tables/stop_metrics.csv` - שורה אחת לכל צומת בגרף, עם
  `stop_id`, `stop_name`, קו רוחב, קו אורך ו-betweenness centrality.
  (מופק על ידי מחברת `04` - מחברת זו אינה מחשבת מחדש צנטרליות.)

### פלט (הכול תחת `outputs/nb/05_critical_station_isolation/`)
- `tables/critical_isolation.csv` - כל תחנה קריטית, מרחקה אל התחנה החלופית הקרובה ביותר,
  ודגל `is_isolated`.
- `tables/isolation_summary.csv` - ספירות רצועות המרחק שמאחורי האיור המרכזי.
- `figures/critical_isolation.png` - **איור המצגת**: תחנות קריטיות מקובצות לפי המרחק
  אל החלופה הקרובה ביותר, בצבע ירוק (ניתנות להחלפה) מול אדום (מבודדות).
- `figures/top_isolated_critical_stops.png` - התחנות המבודדות הבודדות בעלות ה-betweenness
  הגבוה ביותר, בציון שמן.

### מגבלה חשובה (מוצהרת מראש, וחוזרת בסוף)
"קיימת תחנה בטווח 300 מ'" הוא מבחן **מרחבי** בלבד. הוא מתעלם מן השאלה אם אותה תחנה שכנה
משורתת על ידי אותם קווים, באותו כיוון ובתדירות דומה. תחנת אוטובוס מעבר לכביש המשרתת
קו מקומי יחיד אינה תחליף אמיתי למוקד אזורי. לפיכך המבחן **מפריז** במידת ההחלפיות,
ומכאן שמספר התחנות המבודדות המדווח כאן הוא **חסם תחתון** על מספר נקודות הכשל הבודדות
האמיתיות.

## 1. אתחול סביבת העבודה

התא שלהלן מאפשר להריץ את המחברת הן על עותק מקומי של המאגר והן ב-Google Colab. הוא מאתר
את שורש המאגר על ידי טיפוס כלפי מעלה מתיקיית העבודה הנוכחית בחיפוש אחר תיקיית ה-GTFS
`israel-public-transportation`; אם החיפוש נכשל (כלומר, אנו על מכונת Colab חדשה) הוא
משכפל את המאגר. כמו כן מוגדרת `_ensure`, פונקציית עזר המתקינה באמצעות pip רק את החבילות
שאכן חסרות, כך שהרצה חוזרת של המחברת אינה כרוכה בעלות.

`OUT` הוא תיקיית השורש לכל התוצרים שמייצרות המחברות. התיקיות הקיימות `outputs/tables`,
`outputs/figures` ו-`outputs/rail` מכילות את התוצאות המצוטטות בדוח הכתוב, והמחברות הללו
אינן נוגעות בהן.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. ספריות

שלב זה קל במתכוון: אין צורך בספריית גרפים משום שהגרף כבר צומצם לטבלת מדדים ברמת התחנה
במחברת 04. אנו זקוקים ל-`pandas` עבור הטבלה, ל-`numpy` עבור ההמרה לרדיאנים,
ל-`matplotlib` עבור האיורים, ול-`sklearn.neighbors.BallTree` עבור חיפוש השכן הקרוב ביותר.

**מדוע BallTree ולא מטריצת מרחקים בכוח גס?** בגרף ישנן כ-30k תחנות. מטריצת כל-הזוגות
המלאה הייתה מונה 30,000 x 30,000 = 9 x 10^8 מרחקים - כמה ג'יגה-בייטים, ואיטית. עץ כדורים
עם מטריקת `haversine` עונה על השאילתה "התחנה האחרת הקרובה ביותר" עבור כל תחנה בסיבוכיות
O(n log n) ורץ בתוך שניות ספורות.

In [ ]:
_ensure("numpy", "pandas", "matplotlib", "scikit-learn")

import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

print("numpy", np.__version__, "| pandas", pd.__version__)

## 3. הצגת טקסט עברי ב-matplotlib

שמות התחנות ב-GTFS הם בעברית, ואחד האיורים שלהלן מתייג תחנות בודדות לפי שם. matplotlib
אינה מממשת את האלגוריתם הדו-כיווני (bidirectional) של Unicode, ולכן מחרוזות עבריות
משורטטות משמאל לימין ומתקבלות הפוכות. התיקון שלהלן עוטף את `matplotlib.text.Text.set_text`
כך שכל מחרוזת המכילה תווים עבריים מומרת אל סדר התצוגה פעם אחת, לפני השרטוט. התיקון
אידמפוטנטי - הרצה חוזרת של התא לא תחיל אותו פעמיים (והחלה כפולה הייתה הופכת את הטקסט בחזרה).

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## 4. תיקיות השלב ופרמטרי הניתוח

כל מחברת כותבת לתיקיית שלב משלה, כך ששלבים אינם דורסים זה את זה, והבוחן יכול לראות בדיוק
איזה תוצר הופק מאיזה צעד.

שני הפרמטרים המגדירים את הניתוח כולו מרוכזים כאן כדי שיהיה קל לשנותם וקל לערער עליהם:

| קבוע | ערך | משמעות |
| --- | --- | --- |
| `CRITICAL_QUANTILE` | `0.90` | "קריטית" = betweenness בעשירון העליון (top 10%) של כלל התחנות. זו הגדרת העבודה של הפרויקט, בעקביות עם שלב הצנטרליות. |
| `WALK_M` | `300` | מרחק הליכה סביר אל תחנה חלופית. 300 מ' הם בקירוב הליכה של 4 דקות, וזהו סף מקובל בספרות הנגישות לתחבורה ציבורית. |
| `EARTH_M` | `6371000` | רדיוס כדור הארץ הממוצע במטרים - ממיר את הפלט הרדיאני של עץ ה-haversine למטרים. |
| `TOP_N_NAMED` | `15` | כמה תחנות מבודדות בודדות לציין בשמן באיור השני. |

**עלות:** כל מה שנעשה במחברת זו זול - שאילתת עץ הכדורים על כ-30k נקודות אורכת שניות
ספורות וצורכת עשרות בודדות של MB. שום דבר כאן אינו מצריך דגימה או קירוב.

In [ ]:
STAGE = OUT / "05_critical_station_isolation"
TABLES = STAGE / "tables"
FIGURES = STAGE / "figures"
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

CRITICAL_QUANTILE = 0.90      # critical = top 10% betweenness
WALK_M = 300.0                # walking distance to an acceptable alternative stop (metres)
EARTH_M = 6371000.0           # mean Earth radius (metres)
TOP_N_NAMED = 15              # how many isolated stops to name in the second figure

print("Stage folder:", STAGE)

## 5. טעינת מדדי הצנטרליות משלב 04

מחברת זו צורכת את `stop_metrics.csv` משלב הצנטרליות במקום לחשב מחדש את ה-betweenness,
שהוא ללא ספק הגודל היקר ביותר בכל הפרויקט. אם הקובץ חסר, אנו נכשלים בקול רם עם הודעה
מעשית, במקום לייצר בשקט תרשים ריק.

הטוען גם מנרמל כמה וריאנטים של שמות עמודות (`stop_lat`/`lat`,
`approx_betweenness`/`betweenness`) כך שיעבוד עם כל אחת ממוסכמות השמות, ומסיר שורות ללא
קואורדינטות - תחנה נטולת `lat`/`lon` אינה יכולה להשתתף בחיפוש מרחבי. השורות שהוסרו
מדווחות כדי שהאובדן יהיה גלוי ולא מוסתר.

In [ ]:
CENTRALITY_STAGE = OUT / "04_centrality_analysis"
metrics_path = CENTRALITY_STAGE / "tables" / "stop_metrics.csv"

if not metrics_path.exists():
    # tolerate a slightly different stage folder name for the centrality notebook
    candidates = sorted(OUT.glob("*centrality*/tables/stop_metrics.csv"))
    if candidates:
        metrics_path = candidates[0]
    else:
        raise FileNotFoundError(
            f"{metrics_path} missing - run notebook 04 (centrality analysis) first.")

raw = pd.read_csv(metrics_path, encoding="utf-8-sig")

# Normalise column names so either naming convention works.
RENAME = {
    "stop_lat": "lat",
    "stop_lon": "lon",
    "approx_betweenness": "betweenness",
    "betweenness_centrality": "betweenness",
}
raw = raw.rename(columns={k: v for k, v in RENAME.items()
                          if k in raw.columns and v not in raw.columns})

required = {"stop_id", "lat", "lon", "betweenness"}
missing_cols = required - set(raw.columns)
if missing_cols:
    raise KeyError(
        f"{metrics_path} is missing column(s) {sorted(missing_cols)}. "
        f"Found: {sorted(raw.columns)}")

if "stop_name" not in raw.columns:
    raw["stop_name"] = ""

metrics = raw.dropna(subset=["lat", "lon", "betweenness"]).reset_index(drop=True)

print(f"Loaded  : {metrics_path}")
print(f"Stops   : {len(raw):,} rows -> {len(metrics):,} usable "
      f"({len(raw) - len(metrics):,} dropped for missing coordinates/betweenness)")
metrics[["stop_id", "stop_name", "lat", "lon", "betweenness"]].head()

## 6. מרחק אל התחנה החלופית הקרובה ביותר (haversine BallTree)

עבור כל תחנה אנו מבקשים את מרחק המעגל הגדול אל התחנה *האחרת הקרובה ביותר*.

1. קו הרוחב וקו האורך מומרים ל**רדיאנים** - מטריקת `haversine` של `sklearn` מצפה
   לרדיאנים ומחזירה מרחק זוויתי על ספירת היחידה.
2. נבנה `BallTree` מעל אותן נקודות. עץ כדורים מחלק את הנקודות רקורסיבית להיפר-ספירות
   מקוננות, מה שמאפשר לשאילתת השכן הקרוב לגזום ענפים שלמים במקום לסרוק כל נקודה.
3. אנו מבצעים שאילתה עם `k=2`. השכן הקרוב ביותר של נקודה *הוא הנקודה עצמה* (מרחק 0),
   ולכן עמודה `0` נזרקת ועמודה `1` היא התחנה **האחרת** הקרובה ביותר באמת.
4. הכפלת המרחק הזוויתי המוחזר ברדיוס כדור הארץ ממירה אותו למטרים.

יש לשים לב ששני `stop_id` שונים יכולים לחלוק קואורדינטות זהות (שני צדי כביש מקודדים
לעתים לאותה נקודה), מה שמניב מרחק של 0 מ' - תוצאה לגיטימית של "קיימת חלופה ממש כאן"
תחת המבחן המרחבי הזה.

In [ ]:
coords = np.radians(metrics[["lat", "lon"]].to_numpy(dtype=float))

tree = BallTree(coords, metric="haversine")
dist, _ = tree.query(coords, k=2)          # k=2: the stop itself + its nearest neighbour
metrics["nearest_alt_m"] = dist[:, 1] * EARTH_M

print("Distance to nearest other stop (metres), across all graph stops:")
print(metrics["nearest_alt_m"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).round(1))

## 7. פיצול התחנות הקריטיות: ניתנות להחלפה מול מבודדות

"קריטית" מוגדרת כ-**betweenness בגובה האחוזון ה-90 או מעליו** - עשירון התחנות העליון
לפי מספר המסלולים הקצרים העוברים דרכן. מתוכן, תחנה היא **מבודדת** כאשר החלופה הקרובה
ביותר אליה רחוקה מ-`WALK_M` (300 מ'): אובדנה משמעו שלביקוש הסובב אותה אין לאן ללכת בנוחות.

תיקון עקביות קטן ביחס לסקריפט המקורי: סלי ההיסטוגרמה שלהלן סגורים מימין (`right=True`),
כך שתחנה במרחק 300.0 מ' בדיוק נופלת ברצועה הירוקה `100-300 m`, בהתאמה למבחן הבידוד
`> 300 m`. בסקריפט המקורי הסלים היו סגורים משמאל, ולכן תחנה היפותטית במרחק 300.0 מ'
בדיוק הייתה משורטטת באדום אף שנספרה כלא-מבודדת. ההשפעה היא לכל היותר על קבוצה ממידה
אפס של תחנות, אך המספרים בטבלה ובאיור מתיישבים כעת זה עם זה מעצם הבנייה.

In [ ]:
threshold = metrics["betweenness"].quantile(CRITICAL_QUANTILE)
critical = metrics[metrics["betweenness"] >= threshold].copy()
critical["is_isolated"] = critical["nearest_alt_m"] > WALK_M

n_crit = len(critical)
n_isolated = int(critical["is_isolated"].sum())
n_covered = n_crit - n_isolated

print(f"Betweenness threshold (q={CRITICAL_QUANTILE:.2f}) : {threshold:.8f}")
print(f"Critical stops (top 10% betweenness)   : {n_crit:,}")
print(f"  with an alternative within {WALK_M:.0f} m    : {n_covered:,} "
      f"({100 * n_covered / n_crit:.1f}%)")
print(f"  isolated (true single point of fail.) : {n_isolated:,} "
      f"({100 * n_isolated / n_crit:.1f}%)")

## 8. שמירת הטבלאות

נכתבים שני תוצרים:

- **`critical_isolation.csv`** - הרשימה המלאה של התחנות הקריטיות, ממוינת לפי betweenness,
  עם `nearest_alt_m` ו-`is_isolated`. זוהי הראיה ברמת השורה מאחורי כל טענה הנובעת משלב
  זה, וזוהי הטבלה שיש לעיין בה לפני שמכתירים תחנה מסוימת כנקודת תורפה.
- **`isolation_summary.csv`** - חמש רצועות המרחק עם הספירות והשיעורים שלהן; זהו בדיוק
  מה שהאיור המרכזי משרטט, והוא נשמר בנפרד כדי שניתן יהיה לבדוק את האיור מול מספרים
  ולא מול פיקסלים.

שניהם נכתבים בקידוד `utf-8-sig` כדי ששמות תחנות בעברית ייפתחו כראוי ב-Excel.

In [ ]:
BINS = [0, 100, 300, 500, 1000, np.inf]
BAND_LABELS = ["< 100 m", "100-300 m", "300-500 m", "500-1000 m", "> 1 km"]

critical["distance_band"] = pd.cut(critical["nearest_alt_m"], bins=BINS,
                                   labels=BAND_LABELS, right=True, include_lowest=True)

critical_sorted = critical.sort_values("betweenness", ascending=False)
critical_path = TABLES / "critical_isolation.csv"
critical_sorted.to_csv(critical_path, index=False, encoding="utf-8-sig")

band_counts = critical["distance_band"].value_counts().reindex(BAND_LABELS).fillna(0).astype(int)
summary = pd.DataFrame({
    "distance_band": BAND_LABELS,
    "n_critical_stops": band_counts.to_numpy(),
    "share_pct": (100 * band_counts.to_numpy() / n_crit).round(2),
    "verdict": ["substitutable", "substitutable", "isolated", "isolated", "isolated"],
})
summary_path = TABLES / "isolation_summary.csv"
summary.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("wrote", critical_path)
print("wrote", summary_path)
summary

## 9. האיור המרכזי - האם לתחנה הקריטית יש חלופה במרחק הליכה?

זהו האיור המשמש במצגת הסופית. כל עמודה מייצגת רצועה של "המרחק מתחנה קריטית אל התחנה
החלופית הקרובה ביותר אליה", וגובה העמודה הוא מספר התחנות הקריטיות הנופלות באותה רצועה.
הצבע מקודד את המסקנה ולא את הערך: **ירוק** לשתי הרצועות שבתוך סף ההליכה של 300 מ'
(התחנה ניתנת להחלפה, לפחות מבחינה מרחבית) ו**אדום** לשלוש הרצועות שמעבר לו (אין חלופה
במרחק הליכה).

ספירות ואחוזים מודפסים מעל העמודות, משום שעיקרו של התרשים הוא *היחס* בין הירוק לאדום
ולא הגבהים המוחלטים.

In [ ]:
counts = band_counts.to_numpy()
colors = ["#16a34a", "#16a34a", "#dc2626", "#dc2626", "#dc2626"]

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.bar(BAND_LABELS, counts, color=colors, edgecolor="white")

offset = max(counts.max() * 0.02, 1)
for bar, v in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + offset,
            f"{v:,}\n({100 * v / n_crit:.0f}%)", ha="center", va="bottom", fontsize=11)

ax.set_xlabel("Distance to the nearest alternative stop", fontsize=14)
ax.set_ylabel("Number of critical stops", fontsize=14)
ax.set_title("Does a critical stop have an alternative within walking distance?",
             fontsize=16, fontweight="bold", pad=10)
ax.set_ylim(0, counts.max() * 1.18)
ax.tick_params(labelsize=12)
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color="#16a34a", label=f"Alternative within {WALK_M:.0f} m - substitutable in practice"),
    Patch(color="#dc2626", label="Isolated - genuine single point of failure"),
], fontsize=12, loc="upper right")

plt.tight_layout()
fig_path = FIGURES / "critical_isolation.png"
plt.savefig(fig_path, dpi=150)
plt.show()
print("wrote", fig_path)

## 10. אילו תחנות מבודדות חשובות ביותר?

התרשים המצרפי אומר *כמה* תחנות קריטיות מבודדות; הוא אינו אומר *אילו*. איור זה מציין
בשמן את התחנות הקריטיות המבודדות בעלות ה-betweenness הגבוה ביותר - התחנות שבהן עומס
מבני גבוה והיעדר חלופה בת-הליכה מצטלבים. אלה המועמדות הקונקרטיות להשקעה ביתירות
(תוספת שירות מקביל, תחנה שנייה, שאטל).

העמודות מייצגות betweenness (חשיבות מבנית); ההערה על כל עמודה היא המרחק אל התחנה
החלופית הקרובה ביותר, כך ששני חצאי הקריטריון גלויים בו-זמנית. שמות התחנות הם בעברית
ומוצגים דרך תיקון ה-bidi שהותקן בסעיף 3; במקום שבו חסר שם, אנו נסוגים ל-`stop_id`.

In [ ]:
isolated = critical_sorted[critical_sorted["is_isolated"]].copy()

if isolated.empty:
    print("No isolated critical stops found - nothing to plot.")
else:
    top = isolated.head(TOP_N_NAMED).copy()
    top["label"] = (top["stop_name"].fillna("").astype(str).str.strip()
                    .where(lambda s: s != "", top["stop_id"].astype(str)))
    top = top.sort_values("betweenness")   # smallest at the bottom of a barh

    fig, ax = plt.subplots(figsize=(11, 7))
    ax.barh(top["label"], top["betweenness"], color="#dc2626")

    span = top["betweenness"].max()
    for y, (btw, d) in enumerate(zip(top["betweenness"], top["nearest_alt_m"])):
        ax.text(btw + span * 0.01, y, f"{d:,.0f} m to nearest alt.",
                va="center", fontsize=10, color="#374151")

    ax.set_xlim(0, span * 1.28)
    ax.set_xlabel("Betweenness centrality", fontsize=13)
    ax.set_title(f"Top {len(top)} isolated critical stops "
                 f"(top 10% betweenness, no alternative within {WALK_M:.0f} m)",
                 fontsize=15, fontweight="bold", pad=10)
    ax.grid(axis="x", alpha=0.3)
    ax.set_axisbelow(True)

    plt.tight_layout()
    fig2_path = FIGURES / "top_isolated_critical_stops.png"
    plt.savefig(fig2_path, dpi=150)
    plt.show()
    print("wrote", fig2_path)

    display_cols = [c for c in ["stop_id", "stop_name", "betweenness", "nearest_alt_m"]
                    if c in isolated.columns]
    isolated[display_cols].head(TOP_N_NAMED).round(6)

## 11. מגבלות - יש לקרוא לפני ציטוט של כל מספר שלמעלה

**1. מבחן ההחלפיות הוא מרחבי בלבד.** "קיימת תחנה נוספת בטווח 300 מ'" אינו אומר דבר על
השאלה אם אותה תחנה משרתת את *אותם קווים*, ב*אותו כיוון*, ב*תדירות דומה* או באותן שעות.
מסוף אזורי ותחנה מקומית של קו יחיד במרחק 120 מ' נחשבים כאן כברי-החלפה זה בזה, ובבירור
אינם כאלה. לפיכך המבחן **מפריז באופן שיטתי** במידת ההחלפיות של תחנות קריטיות: העמודות
הירוקות גבוהות מדי והעמודות האדומות נמוכות מדי. את מספר התחנות המבודדות יש לקרוא כ**חסם
תחתון** על מספר נקודות הכשל הבודדות האמיתיות. גרסה מחמירה יותר הייתה דורשת שהתחנה
השכנה תחלוק לפחות `route_id` אחד עם התחנה הקריטית.

**2. ה-BallTree נבנה רק מעל תחנות הנוכחות בגרף**, ולא מעל כלל תחנות ה-GTFS. תחנות
שסוננו החוצה בעת בניית הגרף (תחנות מבודדות, תחנות ללא נסיעות שמישות, רשומות כפולות
שהוסרו) אינן נראות לחיפוש השכן הקרוב. אם תחנה כזו ניצבת ממש ליד תחנה קריטית, אותה תחנה
קריטית תסומן כמבודדת אף שבמציאות קיימת תחנה פיזית בסמוך. הטיה זו פועלת בכיוון ההפוך
למגבלה 1, וסדר הגודל שלה אינו מכומת כאן.

**3. מרחק אווירי, לא מרחק הליכה.** haversine מודד מרחק בקו ישר. תחנה במרחק 250 מ' מעבר
לכביש מהיר, למסילת רכבת או לנהר אינה הליכה של 250 מ'. מרחק הליכה אמיתי ברשת יכול רק
להיות ארוך יותר, ולכן גם כאן ההחלפיות נספרת ביתר.

**4. betweenness הוא קירוב.** שלב הצנטרליות שקדם לכאן דוגם צמתי מקור במקום לחשב
betweenness מדויק, ולכן ההשתייכות ל-10% העליונים יציבה עבור המוקדים הדומיננטיים בעליל,
אך רועשת סמוך לסף האחוזון ה-90. אין להתייחס לתחנות היושבות ממש משני צדי הסף כאילו הן
בפנים או בחוץ באופן חד-משמעי.

**5. הספים של 300 מ' ושל 10% העליונים הם מוסכמות, לא ממצאים.** הם נחשפים כקבועים
בסעיף 4 בדיוק כדי שהקורא יוכל להריץ מחדש עם 500 מ' או עם הגדרה של 5% העליונים ולראות
כמה זז היחס המרכזי.

## 12. מסקנות

- **קריטיות בגרף וקריטיות בעולם האמיתי אינן היינו הך.** תחת מבחן החלפיות מרחבי בלבד,
  חלק ניכר מן התחנות שהגרף מסמן כקריטיות מחזיקות תחנה נוספת במרחק הליכה קצר. עבורן,
  סיפור ה"הסר את הצומת והרשת נשברת" של ניתוח הצנטרליות מפריז בהשלכה התפעולית: הנוסעים
  הולכים 2-4 דקות וממשיכים בדרכם. יש להריץ את המחברת כדי לקרוא את הפילוח המדויק מסעיף
  7 - עיקרו של האיור הוא היחס ירוק/אדום, ואת ה*יחס* יש לצטט, לא מספר שנזכר מהרצה קודמת.
- **קבוצת האדום שנותרת היא הממצא האמיתי.** התחנות הקריטיות שאין להן חלופה בטווח 300 מ'
  מהוות רשימה קטנה ומעשית בהרבה מ"10% העליונים לפי betweenness", והקובץ
  `tables/critical_isolation.csv` מציין אותן בשמן. זו הקבוצה שכדאי למקד בה אמצעי יתירות.
- **יש להיות כנים לגבי כיוון השגיאה.** המבחן המרחבי נדיב - הוא סופר כל תחנה סמוכה
  כתחליף, ללא תלות בקווים שהיא משרתת - ולכן קבוצת המבודדות היא רצפה ולא תקרה. ההטיה
  הנגדית (רק תחנות הגרף מצויות בעץ) פועלת בכיוון ההפוך, אך היא קטנה יותר ואינה מכומתת.
  הסיכום הנכון הוא: *ניתוח זה מצמצם את רשימת המועמדות לנקודות כשל בודדות אמיתיות;
  הוא אינו סוגר אותה.*
- **הערת מתודה.** החלק המעניין בשלב זה הוא זול: עץ כדורים עם haversine הופך בעיית מרחקים
  של 9 x 10^8 זוגות לכמה שניות עבודה, וזה מה שהופך את השילוב בין מדד מבני למדד גאוגרפי
  לישים מלכתחילה.